# Data Quality & Deduplication

Raw web-scraped text is noisy. This notebook covers the filters used to build C4, FineWeb, and the Llama training corpus.

| Technique | Purpose | Library |
|---|---|---|
| Heuristic filters | Remove low-quality text | custom |
| Language detection | Keep target language only | `langdetect`, `fasttext` |
| Exact deduplication | Remove identical documents | `hashlib` |
| Near-deduplication | Remove near-duplicate documents | `datasketch` (MinHash LSH) |
| Perplexity filtering | Remove incoherent text | `kenlm` |
| Toxicity scoring | Flag harmful content | `detoxify` |

In [ ]:
%pip install datasketch langdetect datasets tqdm rich matplotlib pandas -q
# detoxify is optional (large download): pip install detoxify

In [ ]:
from pathlib import Path
import json, re, hashlib, random, socket
from datasets import load_dataset
import pandas as pd

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

def is_online(timeout=1.0) -> bool:
    try:
        s = socket.create_connection(('8.8.8.8', 53), timeout=timeout)
        s.close()
        return True
    except OSError:
        return False

# Load web corpus for deduplication practice (with rich multi-noise fallback for offline/classroom execution)
print('Loading web corpus...')
raw_docs = []

if is_online():
    try:
        ds = load_dataset('allenai/c4', 'en', split='train', streaming=True)
        for i, doc in enumerate(ds):
            raw_docs.append({'text': doc['text'], 'url': doc.get('url', '')})
            if i >= 499:
                break
    except Exception as e:
        print(f'Notice: Live HF streaming unavailable ({e}). Using synthetic pretraining corpus.')

if len(raw_docs) < 100:
    # Generate rich synthetic corpus with realistic web noise
    good_articles = [
        'The Transformer architecture, introduced by Vaswani et al. in 2017, relies entirely on self-attention mechanisms to compute representations of inputs and outputs. Unlike recurrent neural networks, transformers process all sequence positions in parallel, significantly accelerating training on modern GPU accelerators.',
        'Scaling laws for large language models, established by Kaplan et al. and refined by Hoffmann et al. (Chinchilla), demonstrate that model performance scales as a power law with compute budget, dataset size, and parameter count when trained under compute-optimal conditions.',
        'Common Crawl provides an open repository of web crawl data that can be accessed and analyzed by anyone. As the foundation of most modern LLM pretraining datasets, rigorous filtering and deduplication are necessary to remove low-quality and repetitive content.',
        'Data deduplication is an essential step in pretraining data preparation. Redundant web pages cause models to disproportionately memorize exact sequences, leading to reduced generalization on downstream NLP tasks.',
        'Mixture of Experts (MoE) architectures decouple parameter count from computational cost by routing each input token to a sparse subset of expert feed-forward networks, enabling models to scale capacity while keeping inference latency manageable.',
        'Low-Rank Adaptation (LoRA) freezes the pretrained model weights and injects trainable rank decomposition matrices into each Transformer layer, drastically reducing the number of trainable parameters for downstream fine-tuning tasks.',
        'Positional encodings provide transformer models with sequence order awareness. Rotary Position Embeddings (RoPE) represent token positions by rotating query and key representations in the complex plane, enabling natural relative position decay.',
        'Grouped-Query Attention (GQA) interpolates between Multi-Head Attention and Multi-Query Attention by sharing key and value heads across multiple query heads, substantially reducing memory bandwidth requirements during autoregressive decoding.',
    ]
    
    # 1. Clean documents
    for i in range(250):
        base = random.choice(good_articles)
        ext = random.choice(good_articles)
        raw_docs.append({
            'text': f'{base} In addition, {ext.lower()} (Article ID: {i})',
            'url': f'https://example-university.edu/research/paper_{i}.html'
        })
    
    # 2. Too short / boilerplate documents
    for i in range(30):
        raw_docs.append({'text': f'Click here to read more. Copyright 2024. All rights reserved. [Snippet {i}]', 'url': f'https://spam.biz/{i}'})
    
    # 3. High symbol / code junk documents
    for i in range(30):
        raw_docs.append({'text': f'var _0x4a=[' + ','.join([f'0x{j:x}' for j in range(25)]) + ']; function _0x1b(){return window;} $(document).ready(function(){console.log("loaded");});', 'url': f'https://script.js/{i}'})
    
    # 4. Repetitive spam documents
    for i in range(30):
        raw_docs.append({'text': 'Best discounts online! ' * 25 + f' Call now {i}!', 'url': f'https://promo.net/{i}'})
    
    # 5. Non-English documents (German, French, Spanish)
    raw_docs.append({'text': 'Die Transformer-Architektur basiert vollstaendig auf Aufmerksamkeitsmechanismen und verzichtet auf rekurrente neuronale Netze fuer die Verarbeitung natuerlicher Sprache.', 'url': 'https://de.wikipedia.org/wiki/Transformer'})
    raw_docs.append({'text': 'Le traitement automatique du langage naturel est un domaine interdisciplinaire de l\'intelligence artificielle et de la linguistique.', 'url': 'https://fr.wikipedia.org/wiki/TAL'})
    raw_docs.append({'text': 'El aprendizaje profundo y los modelos fundacionales han revolucionado el procesamiento del lenguaje natural en los ultimos anos.', 'url': 'https://es.wikipedia.org/wiki/NLP'})
    
    # 6. Exact duplicates
    for i in range(15):
        raw_docs.append(dict(raw_docs[i]))
    
    # 7. Near duplicates (slight typo / minor whitespace difference)
    for i in range(15):
        orig = raw_docs[i]['text']
        raw_docs.append({'text': orig + '   ', 'url': f'https://mirror.org/copy_{i}'})

print(f'Corpus ready: {len(raw_docs)} documents')
print(f'Total characters: {sum(len(d["text"]) for d in raw_docs):,}')
print('\nSample document preview:')
print(raw_docs[0]['text'][:250] + '...')


## 1. Exploratory Analysis — What Does Raw Web Text Look Like?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def doc_stats(text: str) -> dict:
    words = text.split()
    chars = list(text)
    lines = text.strip().split("\n")
    alpha_chars = sum(1 for c in text if c.isalpha())
    return {
        "char_count":    len(text),
        "word_count":    len(words),
        "line_count":    len(lines),
        "alpha_ratio":   alpha_chars / max(len(text), 1),
        "avg_word_len":  sum(len(w) for w in words) / max(len(words), 1),
        "unique_words":  len(set(w.lower() for w in words)),
        "type_token_ratio": len(set(w.lower() for w in words)) / max(len(words), 1),
    }

stats = pd.DataFrame([doc_stats(d["text"]) for d in raw_docs])
print(stats.describe().round(3))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
ax = axes.ravel()

plot_configs = [
    ("word_count",       "Word Count",         "steelblue"),
    ("alpha_ratio",      "Alpha Char Ratio",    "seagreen"),
    ("avg_word_len",     "Avg Word Length",     "darkorange"),
    ("type_token_ratio", "Type-Token Ratio",    "purple"),
    ("line_count",       "Line Count",          "crimson"),
    ("unique_words",     "Unique Words",        "teal"),
]
for axis, (col, label, color) in zip(ax, plot_configs):
    axis.hist(stats[col], bins=40, color=color, alpha=0.8, edgecolor="white")
    axis.set_title(label, fontsize=11)
    axis.set_xlabel(label)
    axis.set_ylabel("Count")

plt.suptitle("Raw Web Corpus — Document Statistics", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 2. Heuristic Quality Filters

The C4 and FineWeb pipelines use fast heuristic rules to remove low-quality documents before any ML-based filtering.

In [ ]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class FilterResult:
    passed: bool
    reason: Optional[str] = None

class HeuristicFilter:
    """Replicates C4 / FineWeb heuristic filtering rules."""

    def __init__(
        self,
        min_words: int = 50,
        max_words: int = 100_000,
        min_alpha_ratio: float = 0.60,
        max_line_duplication_ratio: float = 0.30,
        max_bullet_ratio: float = 0.90,
        max_ellipsis_ratio: float = 0.30,
    ):
        self.min_words = min_words
        self.max_words = max_words
        self.min_alpha_ratio = min_alpha_ratio
        self.max_line_dup_ratio = max_line_duplication_ratio
        self.max_bullet_ratio = max_bullet_ratio
        self.max_ellipsis_ratio = max_ellipsis_ratio

    def filter(self, text: str) -> FilterResult:
        words = text.split()
        lines = [l.strip() for l in text.strip().split("\n") if l.strip()]

        # Word count
        if len(words) < self.min_words:
            return FilterResult(False, f"too_short ({len(words)} words)")
        if len(words) > self.max_words:
            return FilterResult(False, f"too_long ({len(words)} words)")

        # Alphabetic ratio (code/spam has low alpha ratio)
        alpha = sum(1 for c in text if c.isalpha()) / max(len(text), 1)
        if alpha < self.min_alpha_ratio:
            return FilterResult(False, f"low_alpha_ratio ({alpha:.2f})")

        # Duplicate line ratio (menus, tables, repeated nav)
        if lines:
            dup_ratio = 1 - len(set(lines)) / len(lines)
            if dup_ratio > self.max_line_dup_ratio:
                return FilterResult(False, f"high_dup_lines ({dup_ratio:.2f})")

        # Bullet point ratio (listicles, menus)
        if lines:
            bullet_lines = sum(1 for l in lines if l.startswith(("•", "-", "*", "·", "▪")))
            if bullet_lines / len(lines) > self.max_bullet_ratio:
                return FilterResult(False, f"high_bullet_ratio ({bullet_lines/len(lines):.2f})")

        # Ellipsis ratio (truncated content)
        if lines:
            ellipsis_lines = sum(1 for l in lines if l.endswith("..."))
            if ellipsis_lines / len(lines) > self.max_ellipsis_ratio:
                return FilterResult(False, f"high_ellipsis ({ellipsis_lines/len(lines):.2f})")

        # No terminal punctuation (navigation, menus)
        sentences = re.split(r"[.!?]", text)
        long_sentences = [s for s in sentences if len(s.split()) > 3]
        if not long_sentences:
            return FilterResult(False, "no_sentences")

        return FilterResult(True)


# Run on our corpus
hfilter = HeuristicFilter()
filter_results = [hfilter.filter(d["text"]) for d in raw_docs]

passed  = [d for d, r in zip(raw_docs, filter_results) if r.passed]
removed = [r.reason for r in filter_results if not r.passed]

from collections import Counter
print(f"Documents: {len(raw_docs):,}")
print(f"Passed   : {len(passed):,} ({len(passed)/len(raw_docs)*100:.1f}%)")
print(f"Removed  : {len(removed):,} ({len(removed)/len(raw_docs)*100:.1f}%)")
print("\nRemoval reasons:")
for reason, count in Counter(r.split(" (")[0] for r in removed).most_common():
    print(f"  {reason:30s}: {count:4d}")

## 3. Language Detection

In [ ]:
from langdetect import detect, detect_langs, LangDetectException

def detect_language(text: str) -> dict:
    """Detect language with confidence score."""
    try:
        # Use first 500 chars — enough for reliable detection
        sample = text[:500] if len(text) > 500 else text
        langs = detect_langs(sample)
        top = langs[0]
        return {"lang": top.lang, "prob": top.prob, "error": None}
    except LangDetectException as e:
        return {"lang": "unknown", "prob": 0.0, "error": str(e)}

# Test on known samples
test_samples = [
    ("en", "Natural language processing is a subfield of artificial intelligence."),
    ("es", "El procesamiento del lenguaje natural es una rama de la inteligencia artificial."),
    ("fr", "Le traitement du langage naturel est un domaine de l'intelligence artificielle."),
    ("de", "Die Verarbeitung natürlicher Sprache ist ein Teilgebiet der künstlichen Intelligenz."),
    ("zh", "自然语言处理是人工智能的一个子领域。"),
]

print(f"{'Expected':6s} {'Detected':6s} {'Confidence':10s}  Text")
print("-" * 65)
for expected, text in test_samples:
    result = detect_language(text)
    match = "" if result["lang"] == expected else ""
    print(f"{expected:6s} {result['lang']:6s} {result['prob']:10.3f}  {match}  {text[:40]}") 

In [ ]:
# Filter corpus to English only (prob > 0.95)
from tqdm import tqdm

lang_results = [
    detect_language(d["text"])
    for d in tqdm(passed[:500], desc="Detecting languages")
]

english_docs = [
    d for d, lr in zip(passed[:500], lang_results)
    if lr["lang"] == "en" and lr["prob"] > 0.95
]

lang_dist = Counter(lr["lang"] for lr in lang_results)

print(f"Language distribution (top 10 of {len(lang_results)} docs):")
for lang, count in lang_dist.most_common(10):
    pct = count / len(lang_results) * 100
    bar = "█" * int(pct / 2)
    print(f"  {lang:6s} {count:4d} ({pct:5.1f}%) {bar}")

print(f"\nEnglish docs (prob>0.95): {len(english_docs)}")

## 4. Exact Deduplication with SHA-256

In [ ]:
def normalize_for_dedup(text: str) -> str:
    """Normalize text before hashing — catch near-identical docs with minor formatting diffs."""
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-z0-9 ]", "", text)
    return text.strip()

def exact_dedup(docs: list[dict]) -> tuple[list[dict], int]:
    """Remove exact duplicates by hashing normalized text."""
    seen_hashes = set()
    unique_docs = []
    duplicates  = 0

    for doc in docs:
        normalized = normalize_for_dedup(doc["text"])
        h = hashlib.sha256(normalized.encode("utf-8")).hexdigest()

        if h not in seen_hashes:
            seen_hashes.add(h)
            unique_docs.append({**doc, "hash": h})
        else:
            duplicates += 1

    return unique_docs, duplicates

# Inject some synthetic exact duplicates
docs_with_dups = english_docs + english_docs[:50]  # add 50 copies
deduped, n_removed = exact_dedup(docs_with_dups)

print(f"Input docs    : {len(docs_with_dups):,}")
print(f"After dedup   : {len(deduped):,}")
print(f"Duplicates    : {n_removed:,} ({n_removed/len(docs_with_dups)*100:.1f}%)")

## 5. Near-Deduplication with MinHash LSH

Exact hashing misses documents that are 95% the same. MinHash estimates Jaccard similarity between documents in $O(n)$ — without comparing every pair ($O(n^2)$).

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

MinHash approximates this using random hash functions.

In [ ]:
try:
    from datasketch import MinHash, MinHashLSH
except ImportError:
    print('Notice: datasketch library not found. Using educational pure-Python MinHash fallback.')
    import hashlib
    
    class MinHash:
        """Pure-Python educational MinHash implementation."""
        def __init__(self, num_perm: int = 128):
            self.num_perm = num_perm
            self.digest = [0xFFFFFFFF] * num_perm
            
        def update(self, b: bytes):
            for i in range(self.num_perm):
                h = int(hashlib.sha256(f"{i}:".encode() + b).hexdigest()[:8], 16)
                if h < self.digest[i]:
                    self.digest[i] = h
                    
        def jaccard(self, other) -> float:
            matches = sum(1 for a, b in zip(self.digest, other.digest) if a == b)
            return matches / self.num_perm

    class MinHashLSH:
        """Pure-Python educational LSH indexing implementation."""
        def __init__(self, threshold: float = 0.7, num_perm: int = 128):
            self.threshold = threshold
            self.num_perm = num_perm
            self.index = {}
            
        def insert(self, key: str, m: MinHash):
            self.index[key] = m
            
        def query(self, m: MinHash) -> list:
            return [k for k, other in self.index.items() if m.jaccard(other) >= self.threshold]

def text_to_shingles(text: str, k: int = 5) -> set[str]:
    """Convert text to character k-shingles (n-grams)."""
    text = normalize_for_dedup(text)
    return {text[i:i+k] for i in range(len(text) - k + 1)}

def build_minhash(text: str, num_perm: int = 128) -> MinHash:
    m = MinHash(num_perm=num_perm)
    for shingle in text_to_shingles(text):
        m.update(shingle.encode('utf-8'))
    return m

# Demo: compute exact vs MinHash similarity
doc1 = 'Natural language processing enables computers to understand human language.'
doc2 = 'Natural language processing allows computers to comprehend human language.'  # near-duplicate
doc3 = 'Deep learning has revolutionized computer vision and speech recognition systems.'

sh1, sh2, sh3 = text_to_shingles(doc1), text_to_shingles(doc2), text_to_shingles(doc3)

def jaccard(a: set, b: set) -> float:
    return len(a & b) / max(len(a | b), 1)

m1, m2, m3 = build_minhash(doc1), build_minhash(doc2), build_minhash(doc3)

print('Pairwise similarity:')
print(f'  doc1 vs doc2 — exact Jaccard: {jaccard(sh1, sh2):.3f}  MinHash approx: {m1.jaccard(m2):.3f}')
print(f'  doc1 vs doc3 — exact Jaccard: {jaccard(sh1, sh3):.3f}  MinHash approx: {m1.jaccard(m3):.3f}')
print(f'  doc2 vs doc3 — exact Jaccard: {jaccard(sh2, sh3):.3f}  MinHash approx: {m2.jaccard(m3):.3f}')


In [ ]:
# Build the LSH index and deduplicate the corpus
# threshold=0.7 means documents with >70% shingle overlap are considered duplicates

NUM_PERM   = 128
THRESHOLD  = 0.7

def near_dedup_corpus(docs: list[dict], threshold: float = 0.7, num_perm: int = 128) -> tuple[list[dict], list[int]]:
    lsh = MinHashLSH(threshold=threshold, num_perm=num_perm)
    minhashes = []

    # Build index
    for i, doc in enumerate(tqdm(docs, desc="Building MinHash index")):
        m = build_minhash(doc["text"], num_perm=num_perm)
        minhashes.append(m)
        try:
            lsh.insert(str(i), m)
        except ValueError:
            pass  # collision

    # Query for duplicates
    duplicate_indices = set()
    for i, m in enumerate(tqdm(minhashes, desc="Querying duplicates")):
        if i in duplicate_indices:
            continue
        neighbors = lsh.query(m)
        for n in neighbors:
            j = int(n)
            if j != i:
                duplicate_indices.add(j)  # mark the later occurrence as duplicate

    unique = [d for i, d in enumerate(docs) if i not in duplicate_indices]
    return unique, sorted(duplicate_indices)

# Run on English filtered docs
print(f"Before near-dedup: {len(deduped)} unique (exact) docs")
near_deduped, dup_indices = near_dedup_corpus(deduped[:200], threshold=THRESHOLD)
print(f"After near-dedup : {len(near_deduped)} docs")
print(f"Near-duplicates  : {len(dup_indices)} ({len(dup_indices)/len(deduped[:200])*100:.1f}%)")

In [ ]:
# Inspect what was flagged as near-duplicate
if dup_indices:
    for idx in dup_indices[:3]:
        print(f"\n--- Near-duplicate (doc index {idx}) ---")
        print(deduped[idx]["text"][:200])
        print(f"URL: {deduped[idx].get('url', 'n/a')}")

## 6. Text Repetition Filter

Spam and low-quality generated text often has high internal repetition. This is an intra-document dedup filter.

In [ ]:
def repetition_score(text: str, ngram_n: int = 4) -> float:
    """
    Fraction of text covered by repeated n-grams.
    High score = repetitive/spam content.
    Based on FineWeb / Gopher deduplication criteria.
    """
    words = text.lower().split()
    if len(words) < ngram_n * 2:
        return 0.0

    ngrams = [" ".join(words[i:i+ngram_n]) for i in range(len(words) - ngram_n + 1)]
    counts = Counter(ngrams)

    # Words covered by repeated n-grams
    repeated_words = sum((count - 1) * ngram_n for ng, count in counts.items() if count > 1)
    return min(repeated_words / max(len(words), 1), 1.0)

# Test
good_text = "The transformer architecture has revolutionized natural language processing. It uses self-attention mechanisms to capture long-range dependencies in text."
spam_text = "Buy now buy now buy now best deals best deals best deals click here click here click here discount offer discount offer discount offer"

print(f"Good text repetition score: {repetition_score(good_text):.3f}")
print(f"Spam text repetition score: {repetition_score(spam_text):.3f}")

# Apply to corpus
rep_scores = [repetition_score(d["text"]) for d in near_deduped]
low_rep = [d for d, s in zip(near_deduped, rep_scores) if s < 0.1]
print(f"\nDocs after repetition filter (score < 0.1): {len(low_rep)} / {len(near_deduped)}")

## 7. Quality Score Dashboard — Putting It All Together

In [ ]:
def compute_quality_score(text: str) -> dict:
    """Compute a multi-signal quality score for a document."""
    words = text.split()
    alpha = sum(1 for c in text if c.isalpha()) / max(len(text), 1)
    lines = [l.strip() for l in text.strip().split("\n") if l.strip()]
    dup_line_ratio = 1 - len(set(lines)) / max(len(lines), 1)
    rep_score = repetition_score(text)

    # Sentence terminal punctuation ratio
    sentences = re.split(r"[.!?]", text)
    long_sents = [s.strip() for s in sentences if len(s.split()) >= 5]
    punct_ratio = len(long_sents) / max(len(sentences), 1)

    # Composite score: higher is better
    score = (
        0.3 * alpha
        + 0.2 * (1 - dup_line_ratio)
        + 0.2 * (1 - rep_score)
        + 0.2 * punct_ratio
        + 0.1 * min(len(words) / 200, 1.0)  # reward adequate length
    )
    return {
        "alpha_ratio":     alpha,
        "dup_line_ratio":  dup_line_ratio,
        "repetition":      rep_score,
        "punct_ratio":     punct_ratio,
        "word_count":      len(words),
        "quality_score":   score,
    }

# Score all docs and build final filtered corpus
scored_docs = [
    {**d, **compute_quality_score(d["text"])}
    for d in raw_docs[:500]
]
scored_df = pd.DataFrame(scored_docs)

# Keep top 70% by quality score
threshold_score = scored_df["quality_score"].quantile(0.30)
high_quality = scored_df[scored_df["quality_score"] >= threshold_score]

print(f"Documents scored     : {len(scored_df)}")
print(f"High quality (top 70%): {len(high_quality)}")
print(f"\nQuality score stats:")
print(scored_df["quality_score"].describe().round(3))

In [ ]:
# Visualize quality score distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(scored_df["quality_score"], bins=40, color="steelblue", alpha=0.8, edgecolor="white")
axes[0].axvline(threshold_score, color="red", linestyle="--", linewidth=2, label=f"threshold ({threshold_score:.2f})")
axes[0].set_title("Quality Score Distribution")
axes[0].set_xlabel("Quality Score")
axes[0].legend()

# Correlation heatmap of sub-scores
corr_cols = ["alpha_ratio", "dup_line_ratio", "repetition", "punct_ratio", "quality_score"]
corr = scored_df[corr_cols].corr()
im = axes[1].imshow(corr, cmap="RdYlGn", aspect="auto", vmin=-1, vmax=1)
axes[1].set_xticks(range(len(corr_cols)))
axes[1].set_yticks(range(len(corr_cols)))
axes[1].set_xticklabels(corr_cols, rotation=45, ha="right", fontsize=9)
axes[1].set_yticklabels(corr_cols, fontsize=9)
axes[1].set_title("Feature Correlation")
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        axes[1].text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.show()

## 8. Full Pipeline — Putting All Steps Together

In [ ]:
from datasets import Dataset

def run_full_pipeline(docs: list[dict]) -> dict:
    """Run the complete data quality pipeline and return a report."""
    report = {"initial": len(docs)}

    # Step 1: Heuristic filter
    hfilter = HeuristicFilter()
    docs = [d for d in docs if hfilter.filter(d["text"]).passed]
    report["after_heuristic"] = len(docs)

    # Step 2: Language filter (English only)
    docs = [
        d for d in docs
        if detect_language(d["text"])["lang"] == "en"
    ]
    report["after_lang_filter"] = len(docs)

    # Step 3: Exact dedup
    docs, n_exact = exact_dedup(docs)
    report["after_exact_dedup"] = len(docs)
    report["exact_dups_removed"] = n_exact

    # Step 4: Repetition filter
    docs = [d for d in docs if repetition_score(d["text"]) < 0.15]
    report["after_rep_filter"] = len(docs)

    # Step 5: Quality score filter (top 80%)
    if docs:
        scores = [compute_quality_score(d["text"])["quality_score"] for d in docs]
        threshold = sorted(scores)[int(len(scores) * 0.20)]  # bottom 20%
        docs = [d for d, s in zip(docs, scores) if s >= threshold]
    report["after_quality_filter"] = len(docs)

    return docs, report

print("Running full pipeline on 300 docs...")
clean_docs, pipeline_report = run_full_pipeline(raw_docs[:300])

print("\nPipeline Report:")
print(f"{'Step':<30} {'Count':>7} {'Retention':>10}")
print("-" * 50)
initial = pipeline_report['initial']
for step, count in pipeline_report.items():
    if step.startswith('after') or step == 'initial':
        pct = count / initial * 100
        print(f"{step:<30} {count:>7,} {pct:>9.1f}%")

In [ ]:
# Save the cleaned corpus as a HuggingFace Dataset
clean_ds = Dataset.from_list([{"text": d["text"], "url": d.get("url", "")} for d in clean_docs])
clean_ds.save_to_disk(str(DATA_DIR / "clean_corpus"))

print(f"Clean dataset: {clean_ds}")
print(f"Saved to: {DATA_DIR / 'clean_corpus'}")

## 9. Exercise

Take the corpus you built in **3.1-WebScraping-Static** and run it through the full quality pipeline:

1. Apply heuristic filters and report removal reasons
2. Filter to English only
3. Exact dedup with SHA-256
4. Near-dedup with MinHash (threshold=0.7)
5. Score remaining docs and keep the top 80%
6. Save as a HuggingFace Dataset

**Stretch goal:** Build a Pandas DataFrame with all quality signals for your corpus and make scatter plots to visualize what types of documents get filtered.

In [ ]:
# Your code here
# Load your corpus from 3.1
corpus_path = DATA_DIR / "nlp_corpus.jsonl"

if corpus_path.exists():
    with open(corpus_path) as f:
        my_corpus = [json.loads(line) for line in f]
    print(f"Loaded {len(my_corpus)} documents from 3.1 notebook")
    # Run the pipeline
    # my_clean_docs, my_report = run_full_pipeline(my_corpus)
else:
    print("Run 3.1-WebScraping-Static.ipynb first to generate the corpus")

## Summary & Key Takeaways

1. **Data Quality Governs LLM Capability**: Pretraining dataset curation (FineWeb, RedPajama) relies on multi-stage filtering: heuristic filters, line deduplication, and quality classifiers.
2. **MinHash & LSH Deduplication**: MinHash hashing with Locality-Sensitive Hashing identifies near-duplicate web documents in $O(N)$ expected time.
3. **Data Contamination Prevention**: Rigorous n-gram matching and decontamination remove benchmark evaluation questions from pretraining sets.
